In [1]:
# CELL 1 — Setup and Load Processed Data into SQLite

import os
import sys
import sqlite3
import logging
import warnings

import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('..'))

from config.settings import (
    DB_PATH, PROCESSED_DATA_PATH, LOG_PATH,
    COMPANY_NAMES, SUB_SECTOR_MAP
)

logging.basicConfig(
    level = logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(f"../{LOG_PATH}", mode="a"),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger("sql_analysis")

# Load processed datasets
daily_df   = pd.read_csv(f"../{PROCESSED_DATA_PATH}daily_features.csv",
                          parse_dates=["date"])
monthly_df = pd.read_csv(f"../{PROCESSED_DATA_PATH}monthly_summary.csv")
sector_df  = pd.read_csv(f"../{PROCESSED_DATA_PATH}sector_monthly.csv")

# Rename columns to match schema
daily_load = daily_df.rename(columns={
    "open":  "open_price",
    "high":  "high_price",
    "low":   "low_price",
    "close": "close_price"
}).copy()
daily_load["date"] = daily_load["date"].astype(str)

logger.info(f"Loaded daily_df   : {daily_df.shape}")
logger.info(f"Loaded monthly_df : {monthly_df.shape}")
logger.info(f"Loaded sector_df  : {sector_df.shape}")

print("✓ Datasets loaded successfully")

2026-05-26 21:02:25 | INFO | Loaded daily_df   : (19272, 34)
2026-05-26 21:02:25 | INFO | Loaded monthly_df : (924, 26)
2026-05-26 21:02:25 | INFO | Loaded sector_df  : (385, 14)
✓ Datasets loaded successfully


In [4]:
# CELL 2 — Initialize SQLite Database

os.makedirs(os.path.dirname(f'../{DB_PATH}'), exist_ok = True)

conn = sqlite3.connect(f'../{DB_PATH}')
cursor = conn.cursor()

logger.info(f'Connected to: {DB_PATH}')

with open('../sql/schema.sql', 'r') as f:
    schema = f.read()

cursor.executescript(schema)
conn.commit()

logger.info('Schema created - all the tables initialized')
print("✓ Database and schema created")
print("  Tables: dim_ticker, fact_daily, fact_monthly, fact_sector, fact_forecast, ml_metrics")

2026-05-26 21:08:26 | INFO | Connected to: data/market_pulse.db
2026-05-26 21:08:26 | INFO | Schema created - all the tables initialized
✓ Database and schema created
  Tables: dim_ticker, fact_daily, fact_monthly, fact_sector, fact_forecast, ml_metrics


In [6]:
# CELL 3 — Load Dimension Table: dim_ticker

ticker_master = [
    ("AAPL",  "Apple Inc.",             "Technology",  "Consumer Electronics", "NASDAQ", "USD"),
    ("MSFT",  "Microsoft Corporation",  "Technology",  "Enterprise Software",  "NASDAQ", "USD"),
    ("GOOGL", "Alphabet Inc.",          "Technology",  "Internet Services",    "NASDAQ", "USD"),
    ("JPM",   "JPMorgan Chase & Co.",   "Financials",  "Banking",              "NYSE",   "USD"),
    ("GS",    "Goldman Sachs Group",    "Financials",  "Investment Banking",   "NYSE",   "USD"),
    ("BAC",   "Bank of America Corp.",  "Financials",  "Banking",              "NYSE",   "USD"),
    ("JNJ",   "Johnson & Johnson",      "Healthcare",  "Pharmaceuticals",      "NYSE",   "USD"),
    ("PFE",   "Pfizer Inc.",            "Healthcare",  "Biotechnology",        "NYSE",   "USD"),
    ("XOM",   "ExxonMobil Corporation", "Energy",      "Oil & Gas",            "NYSE",   "USD"),
    ("CVX",   "Chevron Corporation",    "Energy",      "Oil & Gas",            "NYSE",   "USD"),
    ("AMZN",  "Amazon.com Inc.",        "Consumer",    "E-Commerce & Cloud",   "NASDAQ", "USD"),
    ("WMT",   "Walmart Inc.",           "Consumer",    "Retail",               "NYSE",   "USD"),
]

cursor.executemany(
    """INSERT OR REPLACE INTO dim_ticker
    (ticker, company_name, sector, sub_sector, exchange, currency)
       VALUES (?,?,?,?,?,?)""",
    ticker_master
)

conn.commit()

logger.info(f'dim_ticker loaded: {len(ticker_master)} rows')
print(f"✓ dim_ticker: {len(ticker_master)} companies loaded")

2026-05-26 21:24:32 | INFO | dim_ticker loaded: 12 rows
✓ dim_ticker: 12 companies loaded


In [8]:
# CELL 4 — Load Fact Tables

#fact daily
daily_load.to_sql('fact_daily', conn, if_exists = 'replace', index = False)
logger.info(f'fact_daily loaded: {len(daily_load):,} rows')

# fact_monthly
monthly_df.to_sql("fact_monthly", conn, if_exists="replace", index=False)
logger.info(f"fact_monthly loaded: {len(monthly_df):,} rows")

# fact_sector
sector_df.to_sql("fact_sector", conn, if_exists="replace", index=False)
logger.info(f"fact_sector loaded: {len(sector_df):,} rows")

conn.commit()

print(f"✓ fact_daily   : {len(daily_load):,} rows")
print(f"✓ fact_monthly : {len(monthly_df):,} rows")
print(f"✓ fact_sector  : {len(sector_df):,} rows")

2026-05-26 21:30:38 | INFO | fact_daily loaded: 19,272 rows
2026-05-26 21:30:38 | INFO | fact_monthly loaded: 924 rows
2026-05-26 21:30:38 | INFO | fact_sector loaded: 385 rows
✓ fact_daily   : 19,272 rows
✓ fact_monthly : 924 rows
✓ fact_sector  : 385 rows


In [12]:
# BUSINESS QUESTION 1:
# Which stocks delivered the best total return over 5 years?
# Which are the best risk-adjusted performers?

q1 = """
SELECT
    f.ticker,
    t.company_name,
    t.sector,
    t.exchange,
    ROUND(MAX(f.cumulative_return), 2) AS total_return_5yr_pct,
    ROUND(AVG(f.close_price), 2) AS avg_close_price,
    ROUND(MAX(CASE
        WHEN f.date = (SELECT MAX(date) FROM fact_daily WHERE ticker = f.ticker)
        THEN f.close_price END), 2) AS latest_price,
    ROUND(AVG(f.volatility_ann) * 100, 2) AS avg_annual_vol_pcvt,
    ROUND(MIN(f.drawdown), 2) AS max_drawdown_pct,
    ROUND(
        MAX(f.cumulative_return) /
        NULLIF(AVG(f.volatility_ann) * 100,0),2) AS return_to_risk_ratio,
    CASE
        WHEN MAX(f.cumulative_return) > 150 AND AVG(f.volatility_ann) * 100 < 30 THEN 'Star Performer'
        WHEN MAX(f.cumulative_return) > 100 THEN 'Strong Performer'
        WHEN MAX(f.cumulative_return) > 50 THEN 'Moderate Performer'
        WHEN MAX(f.cumulative_return) > 0 THEN 'Weak Performer'
        ELSE 'Negative Return'
    END AS performance_category
FROM fact_daily f
JOIN dim_ticker t ON f.ticker = t.ticker
GROUP BY f.ticker, t.company_name, t.sector, t.exchange
ORDER BY total_return_5yr_pct DESC
"""

df_q1 = pd.read_sql_query(q1, conn)
print("=" * 70)
print("QUERY 1 — 5-Year Stock Performance Ranking")
print("=" * 70)
print(df_q1.to_string(index = False))
df_q1.to_csv('../reports/q1_stock_performance.csv', index = False)

QUERY 1 — 5-Year Stock Performance Ranking
ticker           company_name     sector exchange  total_return_5yr_pct  avg_close_price  latest_price  avg_annual_vol_pcvt  max_drawdown_pct  return_to_risk_ratio performance_category
 GOOGL          Alphabet Inc. Technology   NASDAQ                493.20           145.45        382.97              3068.73            -44.32                  0.16     Strong Performer
    GS    Goldman Sachs Group Financials     NYSE                393.14           407.15        996.73              2951.09            -45.62                  0.13     Strong Performer
  AAPL             Apple Inc. Technology   NASDAQ                326.94           170.85        308.82              2861.42            -33.36                  0.11     Strong Performer
   WMT           Walmart Inc.   Consumer     NYSE                269.80            61.50        120.27              2092.00            -25.74                  0.13     Strong Performer
  MSFT  Microsoft Corporation Te

In [16]:
# BUSINESS QUESTION 2:
# Which sectors performed best? How do they compare on risk?

q2 = """
SELECT
    sector,
    COUNT(DISTINCT ticker) AS num_stocks,
    ROUND(AVG(monthly_return), 2) AS avg_monthly_return_cpt,
    ROUND(AVG(monthly_return) * 12, 2) AS approx_annual_return_pct,
    ROUND(AVG(avg_volatility), 2) AS avg_volatility,
    ROUND(MIN(max_drawdown), 4) AS worst_drawdown_pct,
    ROUND(AVG(avg_rsi), 2) AS avg_rsi,
    ROUND(AVG(total_volume), 0) AS avg_monthly_volume,
    ROUND(
        AVG(monthly_return) / NULLIF(AVG(avg_volatility), 0),
        4
    )                                             AS risk_adjusted_score,
    CASE
        WHEN AVG(monthly_return) > 1.0
         AND AVG(avg_volatility) < 0.015 THEN 'High Return / Low Risk'
        WHEN AVG(monthly_return) > 1.0
         AND AVG(avg_volatility) >= 0.015 THEN 'High Return / High Risk'
        WHEN AVG(monthly_return) <= 1.0
         AND AVG(avg_volatility) < 0.015 THEN 'Low Return / Low Risk'
        ELSE 'Low Return / High Risk'
    END                                           AS sector_profile

FROM fact_monthly
GROUP BY sector
ORDER BY approx_annual_return_pct DESC
"""

df_q2 = pd.read_sql_query(q2, conn)
print("=" * 70)
print("QUERY 2 — Sector Performance and Risk Profile")
print("=" * 70)
print(df_q2.to_string(index=False))
df_q2.to_csv("../reports/q2_sector_analysis.csv", index=False)

QUERY 2 — Sector Performance and Risk Profile
    sector  num_stocks  avg_monthly_return_cpt  approx_annual_return_pct  avg_volatility  worst_drawdown_pct  avg_rsi  avg_monthly_volume  risk_adjusted_score          sector_profile
Technology           3                    2.22                     26.62            1.81            -44.3201    54.46         999568655.0               1.2246 High Return / High Risk
  Consumer           2                    1.82                     21.83            1.71            -53.3882    53.74         892590368.0               1.0627 High Return / High Risk
Financials           3                    1.77                     21.22            1.81            -48.9468    54.03         435288088.0               0.9794 High Return / High Risk
    Energy           2                    1.63                     19.61            1.83            -55.0049    52.63         325513807.0               0.8954 High Return / High Risk
Healthcare           2                 

In [17]:
# BUSINESS QUESTION 3:
# How did each sector perform each year?
# Which year was the best and worst for each sector?

q3 = """
SELECT
    sector,
    year,
    ROUND(SUM(avg_daily_return) * 252, 2)    AS annual_return_pct,
    ROUND(AVG(avg_volatility) * 100, 2)      AS avg_vol_pct,
    ROUND(MIN(max_drawdown), 2)              AS worst_drawdown_pct,
    ROUND(AVG(avg_rsi), 1)                   AS avg_rsi,

    CASE
        WHEN SUM(avg_daily_return) * 252 > 20  THEN 'Strong Bull'
        WHEN SUM(avg_daily_return) * 252 > 5   THEN 'Mild Bull'
        WHEN SUM(avg_daily_return) * 252 > -5  THEN 'Flat'
        WHEN SUM(avg_daily_return) * 252 > -20 THEN 'Mild Bear'
        ELSE 'Strong Bear'
    END                                      AS market_regime

FROM fact_sector
GROUP BY sector, year
ORDER BY sector, year
"""

df_q3 = pd.read_sql_query(q3, conn)
print("=" * 70)
print("QUERY 3 — Year-over-Year Sector Returns")
print("=" * 70)
print(df_q3.to_string(index=False))

print("\n--- Annual Return Pivot (%) ---")
pivot = df_q3.pivot(index="sector", columns="year", values="annual_return_pct")
print(pivot.to_string())
df_q3.to_csv("../reports/q3_yoy_returns.csv", index=False)

QUERY 3 — Year-over-Year Sector Returns
    sector  year  annual_return_pct  avg_vol_pct  worst_drawdown_pct  avg_rsi market_regime
  Consumer  2020             510.40       200.58              -22.74     53.3   Strong Bull
  Consumer  2021              34.52       124.07              -16.41     51.3   Strong Bull
  Consumer  2022            -323.77       232.18              -53.39     48.3   Strong Bear
  Consumer  2023             457.25       153.07              -50.91     55.5   Strong Bull
  Consumer  2024             606.26       138.81              -19.49     60.2   Strong Bull
  Consumer  2025             211.91       179.47              -30.88     52.7   Strong Bull
  Consumer  2026             139.51       168.80              -21.74     56.4   Strong Bull
    Energy  2020            -263.37       304.81              -55.00     43.8   Strong Bear
    Energy  2021             571.61       172.19              -36.77     55.0   Strong Bull
    Energy  2022             741.51     

In [20]:
# BUSINESS QUESTION 4:
# Which stocks are riskiest? How are they classified?

q4 = """
SELECT
    f.ticker,
    t.company_name,
    t.sector,

    ROUND(AVG(f.volatility_ann) * 100, 2)    AS annual_vol_pct,
    ROUND(MIN(f.drawdown), 2)                AS max_drawdown_pct,
    ROUND(AVG(f.daily_return), 4)            AS avg_daily_return_pct,

    -- Sharpe approximation (return / risk)
    ROUND(
        (AVG(f.daily_return) * 252) /
        NULLIF(AVG(f.volatility_ann) * 100, 0),
        4
    )                                        AS sharpe_ratio_approx,

    -- Risk tier classification
    CASE
        WHEN AVG(f.volatility_ann) * 100 > 45 THEN 'Very High Risk'
        WHEN AVG(f.volatility_ann) * 100 > 30 THEN 'High Risk'
        WHEN AVG(f.volatility_ann) * 100 > 18 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END                                      AS risk_tier,

    -- Latest RSI
    ROUND(
        AVG(CASE
            WHEN f.date >= DATE('now', '-30 days')
            THEN f.rsi_14 END),
        1
    )                                        AS recent_avg_rsi

FROM fact_daily f
JOIN dim_ticker t ON f.ticker = t.ticker
GROUP BY f.ticker, t.company_name, t.sector
ORDER BY annual_vol_pct DESC
"""

df_q4 = pd.read_sql_query(q4, conn)
print("=" * 70)
print("QUERY 4 — Risk Ranking and Classification")
print("=" * 70)
print(df_q4.to_string(index=False))
df_q4.to_csv("../reports/q4_risk_ranking.csv", index=False)

QUERY 4 — Risk Ranking and Classification
ticker           company_name     sector  annual_vol_pct  max_drawdown_pct  avg_daily_return_pct  sharpe_ratio_approx      risk_tier  recent_avg_rsi
  AMZN        Amazon.com Inc.   Consumer         3357.43            -53.39                0.0891               0.0067 Very High Risk            69.0
 GOOGL          Alphabet Inc. Technology         3068.73            -44.32                0.1286               0.0106 Very High Risk            76.8
   BAC  Bank of America Corp. Financials         2986.98            -48.95                0.0555               0.0047 Very High Risk            43.0
   XOM ExxonMobil Corporation     Energy         2969.08            -55.00                0.0881               0.0075 Very High Risk            52.1
    GS    Goldman Sachs Group Financials         2951.09            -45.62                0.1208               0.0103 Very High Risk            57.7
  AAPL             Apple Inc. Technology         2861.42        

In [21]:
# BUSINESS QUESTION 5:
# Which stocks have the strongest momentum right now?

q5 = """
WITH monthly_momentum AS (
    SELECT
        ticker,
        sector,
        year_month,
        year,
        month,
        close_price,
        monthly_return,
        mom_pct_change,
        avg_rsi,
        monthly_volatility,
        golden_cross,

        -- 3-month rolling average return (momentum indicator)
        ROUND(
            AVG(monthly_return) OVER (
                PARTITION BY ticker
                ORDER BY year, month
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ), 4
        ) AS rolling_3m_return,

        -- Rank within sector by return each month
        RANK() OVER (
            PARTITION BY sector, year, month
            ORDER BY monthly_return DESC
        ) AS sector_rank

    FROM fact_monthly
)

SELECT
    ticker,
    sector,
    year_month,
    ROUND(close_price, 2) AS close_price,
    ROUND(monthly_return, 2) AS monthly_return_pct,
    ROUND(mom_pct_change, 2) AS mom_change_pct,
    ROUND(rolling_3m_return, 2) AS rolling_3m_momentum,
    ROUND(avg_rsi, 1) AS avg_rsi,
    sector_rank,
    CASE WHEN golden_cross = 1 THEN 'Golden Cross' ELSE 'Death Cross' END AS ma_signal,
    CASE
        WHEN rolling_3m_return > 8 THEN 'Strong Uptrend'
        WHEN rolling_3m_return > 2 THEN 'Mild Uptrend'
        WHEN rolling_3m_return > -2 THEN 'Sideways'
        WHEN rolling_3m_return > -8 THEN 'Mild Downtrend'
        ELSE 'Strong Downtrend'
    END AS trend_label
FROM monthly_momentum
ORDER BY year_month DESC, ticker
LIMIT 72
"""

df_q5 = pd.read_sql_query(q5, conn)
print("=" * 70)
print("QUERY 5 — Momentum Analysis (Latest 72 Records)")
print("=" * 70)
print(df_q5.to_string(index=False))
df_q5.to_csv("../reports/q5_momentum.csv", index=False)

QUERY 5 — Momentum Analysis (Latest 72 Records)
ticker     sector year_month  close_price  monthly_return_pct  mom_change_pct  rolling_3m_momentum  avg_rsi  sector_rank    ma_signal      trend_label
  AAPL Technology    2026-05       308.82               13.18           13.91                 5.41     77.2            1 Golden Cross     Mild Uptrend
  AMZN   Consumer    2026-05       266.32                0.60            0.48                 8.25     64.8            1 Golden Cross   Strong Uptrend
   BAC Financials    2026-05        51.80               -3.01           -3.11                 1.65     39.4            3  Death Cross         Sideways
   CVX     Energy    2026-05       191.43                0.17           -0.07                 1.40     52.7            2 Golden Cross         Sideways
 GOOGL Technology    2026-05       382.97               -0.27           -0.48                 7.35     75.6            3 Golden Cross     Mild Uptrend
    GS Financials    2026-05       996.73     

In [22]:
# BUSINESS QUESTION 6:
# Which stocks show actionable technical signals TODAY?

q6 = """
WITH latest_data AS (
    SELECT
        ticker,
        date,
        close_price,
        ma_20, ma_50, ma_200,
        rsi_14, rsi_zone,
        volatility_30d,
        drawdown,
        above_ma50, above_ma200,
        golden_cross,
        bb_upper, bb_lower, bb_pct,
        ROW_NUMBER() OVER (
            PARTITION BY ticker ORDER BY date DESC
        ) AS rn
    FROM fact_daily
)

SELECT
    l.ticker,
    t.company_name,
    t.sector,
    l.date AS as_of_date,
    ROUND(l.close_price, 2) AS price,

    -- Trend
    ROUND(l.ma_50,  2)                  AS ma_50,
    ROUND(l.ma_200, 2)                  AS ma_200,
    CASE WHEN l.above_ma50  = 1 THEN 'Above' ELSE 'Below' END AS vs_ma50,
    CASE WHEN l.above_ma200 = 1 THEN 'Above' ELSE 'Below' END AS vs_ma200,
    CASE WHEN l.golden_cross = 1
         THEN 'Golden Cross' ELSE 'Death Cross' END AS ma_cross,

    -- Momentum
    ROUND(l.rsi_14, 1) AS rsi,
    l.rsi_zone,

    -- Risk
    ROUND(l.volatility_30d * 100, 2) AS vol_30d_pct,
    ROUND(l.drawdown, 2) AS drawdown_pct,

    -- Bollinger position
    CASE
        WHEN l.close_price > l.bb_upper THEN 'Above Upper Band'
        WHEN l.close_price < l.bb_lower THEN 'Below Lower Band'
        ELSE 'Within Bands'
    END AS bb_position,

    -- Composite signal
    CASE
        WHEN l.golden_cross = 1
         AND l.rsi_14 BETWEEN 45 AND 65
         AND l.above_ma200 = 1 THEN 'BULLISH'
        WHEN l.golden_cross = 0
         AND l.rsi_14 > 65 THEN 'BEARISH'
        WHEN l.rsi_14 < 30 THEN 'OVERSOLD - WATCH'
        ELSE 'NEUTRAL'
    END AS signal

FROM latest_data l
JOIN dim_ticker t ON l.ticker = t.ticker
WHERE l.rn = 1
ORDER BY l.close_price DESC
"""

df_q6 = pd.read_sql_query(q6, conn)
print("=" * 70)
print("QUERY 6 — Technical Signal Scanner (Latest Date)")
print("=" * 70)
print(df_q6.to_string(index=False))
df_q6.to_csv("../reports/q6_signals.csv", index=False)

QUERY 6 — Technical Signal Scanner (Latest Date)
ticker           company_name     sector as_of_date  price  ma_50  ma_200 vs_ma50 vs_ma200     ma_cross  rsi   rsi_zone  vol_30d_pct  drawdown_pct      bb_position  signal
    GS    Goldman Sachs Group Financials 2026-05-22 996.73 896.07  843.58   Above    Above Golden Cross 73.7 Overbought       179.74          0.00 Above Upper Band NEUTRAL
  MSFT  Microsoft Corporation Technology 2026-05-22 418.57 399.61  458.27   Above    Below  Death Cross 55.1    Bullish       193.44        -22.29     Within Bands NEUTRAL
 GOOGL          Alphabet Inc. Technology 2026-05-22 382.97 341.14  295.96   Above    Above Golden Cross 49.8    Neutral       236.17         -4.88     Within Bands BULLISH
  AAPL             Apple Inc. Technology 2026-05-22 308.82 270.36  261.09   Above    Above Golden Cross 91.1 Overbought       140.07          0.00     Within Bands NEUTRAL
   JPM   JPMorgan Chase & Co. Financials 2026-05-22 306.38 301.22  303.26   Above    Above 

In [23]:
conn.close()
logger.info("SQL analysis complete — all 6 queries saved")
print("\n✓ Database connection closed")
print("✓ All query results saved to reports/")
print("\nNotebook 04 complete. Proceed to 05_ml_forecasting.ipynb")

2026-05-27 07:32:44 | INFO | SQL analysis complete — all 6 queries saved

✓ Database connection closed
✓ All query results saved to reports/

Notebook 04 complete. Proceed to 05_ml_forecasting.ipynb
